### Step 1: Environment Setup & Source Path Definition
* Imports required PySpark SQL functions (`col`, `current_timestamp`, `current_date`).
* Defines the absolute path to the raw dataset stored inside the Unity Catalog Volume.

In [0]:
from pyspark.sql.functions import input_file_name, current_timestamp, current_date, col

raw_file_path = "/Volumes/dbr_dev_ua5816bd/natalkamartinuk55/raw_data/USvideos.csv"


### Step 2: Read Raw CSV Dataset
* Uses PySpark DataFrameReader to parse the multiline CSV file.
* Enables header detection, automatic schema inference, and quote escape handling for complex video descriptions.

In [0]:
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", "\"")
    .load(raw_file_path)
)

### Step 3: Append Technical Metadata Columns
Enriches the raw records with audit metadata required for Bronze layer compliance:
* `_source_file`: Extracted via Unity Catalog compliant `col("_metadata.file_name")`.
* `_ingestion_timestamp`: Current UTC timestamp of the ingestion process.
* `_load_date`: Current processing date for partitioning and traceability.

In [0]:


df_bronze = (
    df_raw
    .withColumn("_source_file", col("_metadata.file_name"))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_load_date", current_date())
)

display(df_bronze.limit(10))


### Step 4: Idempotent Delta Table Write & Verification
* Writes the transformed DataFrame to Delta format using `mode("overwrite")` to guarantee idempotency across automated job executions.
* Saves the dataset to `dbr_dev_ua5816bd.natalkamartinuk55_bronze.youtube_videos_bronze`.
* Reads back and displays top records to verify data integrity.

In [0]:

target_table = "dbr_dev_ua5816bd.natalkamartinuk55_bronze.youtube_videos_bronze"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_table)
)

display(spark.table(target_table).limit(10))